# Precomputed Lars Forward Return

Load precomputed order-book feature/return parquet files, infer feature columns from the file schema, and fit a Lars path model on the forward-return target.

In [1]:
%load_ext autoreload
%autoreload 2

In [5]:
from __future__ import annotations

import re
import sys
from pathlib import Path

import numpy as np
import polars as pl
from matplotlib import pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ROOT_STR = str(ROOT)
if ROOT_STR not in sys.path:
    sys.path.append(ROOT_STR)

from tools.data import DataSource, DateFrame, Raw, expand_dates
from tools.filters import intraday_time, level_taken, tight_spread, trade_size
from tools.model import LarsAdapter, RidgeAdapter
from tools.pipeline import Pipeline
from tools.score import get_unit_pnl, rmse
from tools.search import grid
from tools.transform import Standardizer

In [14]:
def divide_dates(*args):
    dates = []
    for i in range(1, len(args)):
        dates.append(
            expand_dates(
                f"{args[i - 1]}-{args[i]}",
                end_date=False if i < len(args) - 1 else True,
            )
        )
    return dates

In [15]:
# Data
PROD = "ES"
ROLLING_DATES = divide_dates(20260323, 20260410, 20260425, 20260510, 20260524)
TEST_DATES = expand_dates("20260525-20260529")
L2_DEPTH = 5
MODEL_BATCH_SIZE = 1_000_000
POLARS_ENGINE = "streaming"
FEATURE_RETURN_PATH = str(
    ROOT
    / f"data/orderbook_feature_return_parquet/{{prod}}M6_{{d}}_{{tag}}_{{prod_s}}_full_day_l2_d{L2_DEPTH}_features_return.parquet"
)
REGULAR_HOURS_START = "09:30"
REGULAR_HOURS_END = "16:00"
REGULAR_HOURS_TZ = "America/New_York"

TARGET = "forward_mid_return_bps"
TEST_PNL_THRESHOLD = 0.2

# Lars/search knobs
SAMPLER = "grid"
PATH_EVAL_ALPHAS = np.geomspace(1e-2, 20, 30).tolist()
N_TRIALS = len(PATH_EVAL_ALPHAS)
SEARCH_SPACE = {"alpha": grid(PATH_EVAL_ALPHAS)}

UNDEF_PRICE = 9_223_372_036_854_775_807
TICKSIZE = 250000000

In [16]:
BOOK_COL_RE = re.compile(r"^(?:bid|ask)_(?:px|sz|ct)_\d+$")
SCHEMA_NON_FEATURE_COLS = {
    "date",
    "nature",
    "ts_event",
    "ts_recv",
    "symbol",
    "instrument_id",
    "row_nr",
    "sequence",
    "publisher_id",
    "trade_px",
    "trade_px_last",
    "trade_vwap",
    "trade_sz",
    "trade_side",
    "trade_count",
    "trade_fills",
    "trade_levels",
    "trade_posted_sz",
    "buy_trade_sz",
    "sell_trade_sz",
    "buy_trade_count",
    "sell_trade_count",
}


def infer_features_from_schema(schema: pl.Schema, target: str = TARGET) -> list[str]:
    features = []
    for col in schema.names():
        if col == target or col in SCHEMA_NON_FEATURE_COLS or BOOK_COL_RE.match(col):
            continue
        features.append(col)
    if not features:
        raise ValueError("no feature columns inferred from parquet schema")
    return features


FEATURE_SCHEMA_PATH, _ = Raw.resolve_path(ROLLING_DATES[0][0], PROD, FEATURE_RETURN_PATH)
FEATURE_SCHEMA = pl.scan_parquet(FEATURE_SCHEMA_PATH).collect_schema()
FEATURES = infer_features_from_schema(FEATURE_SCHEMA)
META_COLS = [col for col in FEATURE_SCHEMA.names() if col not in FEATURES and col != TARGET]
LOAD_COLS = list(dict.fromkeys([*META_COLS, *FEATURES, TARGET]))

FEATURES

['imb_d1',
 'imb_d3',
 'imb_d5',
 'weighted_price_sz2',
 'weighted_price_sz5',
 'weighted_price_sz10',
 'trade_momentum_hl1s',
 'push_momentum_hl1s',
 'pull_momentum_hl1s',
 'trade_corr_side_hl1s',
 'trade_corr_volume_hl1s',
 'log_return_hl1s',
 'ewma_spread_hl1s',
 'trade_momentum_hl10s',
 'push_momentum_hl10s',
 'pull_momentum_hl10s',
 'trade_corr_side_hl10s',
 'trade_corr_volume_hl10s',
 'log_return_hl10s',
 'ewma_spread_hl10s',
 'trade_momentum_hl30s',
 'push_momentum_hl30s',
 'pull_momentum_hl30s',
 'trade_corr_side_hl30s',
 'trade_corr_volume_hl30s',
 'log_return_hl30s',
 'ewma_spread_hl30s',
 'trade_momentum_hl120s',
 'push_momentum_hl120s',
 'pull_momentum_hl120s',
 'trade_corr_side_hl120s',
 'trade_corr_volume_hl120s',
 'log_return_hl120s',
 'ewma_spread_hl120s']

In [17]:
VALID_ROWS = (
    (pl.col("bid_px_0") != UNDEF_PRICE)
    & (pl.col("ask_px_0") != UNDEF_PRICE)
    & (pl.col("ask_px_0") > pl.col("bid_px_0"))
    & pl.col(TARGET).is_not_null()
    & pl.all_horizontal([pl.col(c).is_finite() for c in FEATURES])
)
REGULAR_HOURS = intraday_time(REGULAR_HOURS_START, REGULAR_HOURS_END, timezone=REGULAR_HOURS_TZ)
TIGHT_SPREAD = tight_spread(TICKSIZE)
VALID_REGULAR_ROWS = VALID_ROWS & REGULAR_HOURS & TIGHT_SPREAD
TRAIN_ROWS = VALID_REGULAR_ROWS & (level_taken() | trade_size(0.3))

REGULAR_HOURS

<Expr ['[([(col("ts_event").dt.convert…'] at 0x7605987B7D10>

In [18]:
def load_feature_return_date(day: str, prod: str = PROD) -> DateFrame:
    return Raw.load_date(day, prod, path=FEATURE_RETURN_PATH, cols=LOAD_COLS)


def regular_loader(dates: list[str]) -> list[DateFrame]:
    return [load_feature_return_date(day) for day in dates]

In [19]:
FEATURE_TEST_SCORE = get_unit_pnl(TEST_PNL_THRESHOLD)
FEATURE_TEST_SCORE_DESCENDING = True

test_date_src = DataSource(
    dates=TEST_DATES,
    loader=regular_loader,
    target=TARGET,
    features=FEATURES,
    filters=(VALID_REGULAR_ROWS,),
    polars_engine=POLARS_ENGINE,
)

feature_test_states = dict.fromkeys(FEATURES)
feature_test_rows = 0
for x, y_true, ctx in test_date_src.batches(MODEL_BATCH_SIZE):
    feature_test_rows += int(ctx["n"])
    for idx, feature in enumerate(FEATURES):
        feature_test_states[feature] = FEATURE_TEST_SCORE(
            y_true,
            x[:, idx],
            ctx,
            combine_with=feature_test_states[feature],
        )

feature_test_scores = (
    pl.DataFrame(
        [
            {
                "feature": feature,
                "score": getattr(FEATURE_TEST_SCORE, "__name__", "score"),
                "test_score": float(state),
                "score_n": int(getattr(state, "n", 0)),
                "rows": feature_test_rows,
            }
            for feature, state in feature_test_states.items()
            if state is not None
        ]
    )
    .sort("test_score", descending=FEATURE_TEST_SCORE_DESCENDING)
)

feature_test_scores

Loading data: 25.6Mrow [00:24, 1.06Mrow/s]


feature,score,test_score,score_n,rows
str,str,f64,i64,i64
"""weighted_price_sz2""","""unit_pnl_0.2""",0.334565,450,25573459
"""trade_corr_volume_hl30s""","""unit_pnl_0.2""",0.192104,397137,25573459
"""pull_momentum_hl30s""","""unit_pnl_0.2""",0.143328,1593862,25573459
"""imb_d5""","""unit_pnl_0.2""",0.110949,19634992,25573459
"""imb_d3""","""unit_pnl_0.2""",0.107308,20883906,25573459
…,…,…,…,…
"""pull_momentum_hl120s""","""unit_pnl_0.2""",-0.136467,293383,25573459
"""push_momentum_hl30s""","""unit_pnl_0.2""",-0.178692,1813061,25573459
"""weighted_price_sz5""","""unit_pnl_0.2""",-0.17875,2645,25573459


In [23]:
pipeline = Pipeline(
    rolling_dates=ROLLING_DATES,
    test_dates=TEST_DATES,
    adapter=LarsAdapter(
        batch_size=MODEL_BATCH_SIZE,
        alpha_min=0.0,
        fit_intercept=False,
        path_eval_alphas=PATH_EVAL_ALPHAS,
        stats_backend="gpu",
    ),
    # adapter=RidgeAdapter(batch_size=MODEL_BATCH_SIZE, fit_intercept=False, stats_backend="gpu"),
    target=TARGET,
    features=FEATURES,
    data_loader=regular_loader,
    search_space=SEARCH_SPACE,
    val_score=rmse,
    transform=Standardizer(FEATURES, batch_size=MODEL_BATCH_SIZE),
    train_filters=(TRAIN_ROWS,),
    val_filters=(VALID_REGULAR_ROWS,),
    test_filters=(VALID_REGULAR_ROWS,),
    sampler=SAMPLER,
    n_trials=N_TRIALS,
    cache_arrays=False,
    score_direction="minimize",
    polars_engine=POLARS_ENGINE,
)
pipeline

Pipeline(rolling_dates=[['2026-03-23', '2026-03-24', '2026-03-25', '2026-03-26', '2026-03-27', '2026-03-30', '2026-03-31', '2026-04-01', '2026-04-02', '2026-04-06', '2026-04-07', '2026-04-08', '2026-04-09'], ['2026-04-10', '2026-04-13', '2026-04-14', '2026-04-15', '2026-04-16', '2026-04-17', '2026-04-20', '2026-04-21', '2026-04-22', '2026-04-23', '2026-04-24'], ['2026-04-27', '2026-04-28', '2026-04-29', '2026-04-30', '2026-05-01', '2026-05-04', '2026-05-05', '2026-05-06', '2026-05-07', '2026-05-08'], ['2026-05-11', '2026-05-12', '2026-05-13', '2026-05-14', '2026-05-15', '2026-05-18', '2026-05-19', '2026-05-20', '2026-05-21', '2026-05-22']], test_dates=['2026-05-26', '2026-05-27', '2026-05-28', '2026-05-29'], adapter=LarsAdapter(alpha=1.0, method='lasso', fit_intercept=False, max_iter=500, alpha_min=0.0, eps=2.220446049250313e-16, positive=False, batch_size=1000000, max_features=None, stats_backend='gpu', stats_dtype=<class 'numpy.float64'>, streaming=True, cache_path=True, vectorized_p

In [ ]:
train_result = pipeline.train(verbose=2)
train_result

[I 2026-07-06 23:32:07,491] A new study created in memory with name: no-name-8b030d52-7cac-49bd-8379-7ec1d290c989


======== Optuna study created. Launching optimization.
======== running params {'alpha': 0.016891073086154205}
======== fold: 0, with train = ['2026-03-23', '2026-03-24', '2026-03-25', '2026-03-26', '2026-03-27', '2026-03-30', '2026-03-31', '2026-04-01', '2026-04-02', '2026-04-06', '2026-04-07', '2026-04-08', '2026-04-09'] and val = ['2026-04-10', '2026-04-13', '2026-04-14', '2026-04-15', '2026-04-16', '2026-04-17', '2026-04-20', '2026-04-21', '2026-04-22', '2026-04-23', '2026-04-24']


Loading data: 7.08Mrow [01:17, 91.4krow/s]
Loading data: 7.08Mrow [00:55, 127krow/s] 
Loading data: 85.1Mrow [02:39, 533krow/s]


======== loss = 1.8923149536678436, running average = 1.8923149536678436
======== fold: 1, with train = ['2026-03-23', '2026-03-24', '2026-03-25', '2026-03-26', '2026-03-27', '2026-03-30', '2026-03-31', '2026-04-01', '2026-04-02', '2026-04-06', '2026-04-07', '2026-04-08', '2026-04-09', '2026-04-10', '2026-04-13', '2026-04-14', '2026-04-15', '2026-04-16', '2026-04-17', '2026-04-20', '2026-04-21', '2026-04-22', '2026-04-23', '2026-04-24'] and val = ['2026-04-27', '2026-04-28', '2026-04-29', '2026-04-30', '2026-05-01', '2026-05-04', '2026-05-05', '2026-05-06', '2026-05-07', '2026-05-08']


Loading data: 7.91Mrow [00:47, 160krow/s]

In [ ]:
test_result = pipeline.test(get_unit_pnl(0.0))
test_result

In [ ]:
pnl_result = pipeline.test(get_unit_pnl(0.1, power=2))
pnl_result

In [ ]:
model = pipeline.get_model()
if model.coefs_path is None:
    raise RuntimeError("Run pipeline.train() before plotting the coefficient path.")

for i, feature in enumerate(FEATURES):
    plt.plot(model.alphas, model.coefs_path[i, :], label=feature)
plt.xscale("log")
plt.xlabel("alpha")
plt.ylabel("standardized coefficient")
plt.legend()

In [ ]:
print(np.mean(pnl_result["y_pred"]), np.std(pnl_result["y_pred"]))
_ = plt.hist(pnl_result["y_pred"], bins=100, log=True, density=False)
plt.xlabel("prediction")
plt.ylabel("count")

In [ ]:
model = pipeline.get_model()
transform = pipeline.fitted_transform

if model is None or model.coef is None:
    raise RuntimeError("Run pipeline.train() before inspecting coefficients.")
if transform is None or transform.means is None or transform.stds is None:
    raise RuntimeError("Run pipeline.train() with a fitted Standardizer before inspecting real coefficients.")

features = list(pipeline.features)
raw_coef = [float(coef) for coef in model.coef]
means = [float(transform.means[feature]) for feature in features]
stds = [float(transform.stds[feature]) for feature in features]
real_coef = [coef / std for coef, std in zip(raw_coef, stds)]

raw_intercept = float(model.intercept)
real_intercept = raw_intercept - sum(
    coef * mean / std for coef, mean, std in zip(raw_coef, means, stds)
)

df_coef = pl.DataFrame(
    [
        {
            "term": "intercept",
            "raw_coef": raw_intercept,
            "real_coef": real_intercept,
            "standardizer_mean": None,
            "standardizer_std": None,
        },
        *[
            {
                "term": feature,
                "raw_coef": raw,
                "real_coef": real,
                "standardizer_mean": mean,
                "standardizer_std": std,
            }
            for feature, raw, real, mean, std in zip(features, raw_coef, real_coef, means, stds)
        ],
    ]
)

with pl.Config(tbl_rows=-1):
    display(df_coef)

In [ ]:
pipeline.save_pipeline("./dump/")